<div style="text-align: justify; max-width: 90ch">

# Worked example: a paper assistant on Kaggle

**Status: Optional / career-track.**

This is the worked example of the course's final assignment: a question-answering assistant over 23 papers from the [ACL Anthology](https://aclanthology.org/) (2025) on evaluating LLM agents and LLM judges.
It answers from the papers, names the papers it used, and is evaluated with MLflow, all on a free Kaggle GPU.

Copy it, replace the papers with papers from your own field, and adapt it.

**What this notebook covers**

- **Setup**: Ollama, PostgreSQL with pgvector and the Python packages on a fresh Kaggle machine
- **Corpus**: papers whose license allows republishing, read from PDF
- **Index**: chunks and embeddings in pgvector (`lec_02e`)
- **Cited answers**: a pydantic schema for the answer and its papers (`lec_01f`)
- **Agentic graph**: search again or refuse, with LangGraph (`lec_03e`)
- **Evaluation**: two variants, deterministic scorers and an LLM judge (`lec_03a`, `lec_03b`), the judge checked against hand labels (`lec_03c`)
- **Decision record**: the model choice, requirements first (`lec_01b`)

</div>

<div style="text-align: justify; max-width: 90ch">

**Run it on Kaggle.** In the notebook editor's side panel:

- **Accelerator**: GPU T4 x2
- **Internet**: on (needs a phone-verified Kaggle account)
- **Input**: the dataset with the PDFs, `papers.csv`, `eval_set.jsonl` and `judge_labels.jsonl`

Then *Save Version* with *Save & Run All*. In one example run the whole notebook took about 20 minutes, a quarter of it in setup.
The saved version keeps every output, so visitors read the results without running anything.

</div>

In [ ]:
# --- Settings: the cell to edit when you adapt this notebook ---
SMALL_MODEL = "qwen3:1.7b"  # variant 1
LARGE_MODEL = "qwen3:8b"  # variant 2
JUDGE_MODEL = "qwen3:14b"  # larger than both variants, and not one of them
EMBED_MODEL = "nomic-embed-text"
OLLAMA_VERSION = "0.34.0"  # pinned: a new release can change answers and speed
TOP_K = 4  # chunks per search
ALLOWED_LICENSES = ["CC BY 4.0", "CC BY-SA 4.0", "CC0 1.0"]
MLFLOW_DB = "/kaggle/working/mlflow.db"  # /kaggle/working is saved with the version
EXPERIMENT = "paper-assistant"

<div style="text-align: justify; max-width: 90ch">

## 1. A fresh machine every session

Kaggle starts every session on a new machine: nothing you installed last time is still there. On your laptop the course's guides installed everything once;
here the notebook installs it at the top, every run.

The next cell installs Ollama (guide `01c_ollama`), starts its server in the background and downloads the four models. The downloads are about 16 GB and take a few minutes.

</div>

In [ ]:
%%bash -s "$OLLAMA_VERSION"
apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null  # Ollama's archive format
curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=$1 sh > /dev/null 2>&1
pgrep -x ollama > /dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
until ollama list > /dev/null 2>&1; do sleep 1; done
for model in nomic-embed-text qwen3:1.7b qwen3:8b qwen3:14b; do
  ollama pull "$model" > /dev/null 2>&1 && echo "pulled $model"
done
ollama --version

<div style="text-align: justify; max-width: 90ch">

Kaggle has no Docker, so PostgreSQL and pgvector come from the [PostgreSQL apt repository](https://www.postgresql.org/download/linux/ubuntu/) instead of the container of `lec_02e`.
The database has no password at all:

- **Peer authentication**: PostgreSQL lets a local system user in as the database role of the same name, checked by the operating system.
  The notebook runs as `root`, so it connects as the role `root`.
- **Least privilege**: `root` owns the `papers` database but is not a superuser; the `postgres` superuser installs the `vector` extension once.

</div>

In [ ]:
%%bash
apt-get -qq install -y curl ca-certificates lsb-release > /dev/null
install -d /usr/share/postgresql-common/pgdg
curl -fsS -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-17 postgresql-17-pgvector > /dev/null 2>&1
pg_lsclusters | grep -q online || pg_ctlcluster 17 main start
su postgres -c "createuser root" 2> /dev/null || true  # already there on a second run
su postgres -c "createdb --owner=root papers" 2> /dev/null || true
su postgres -c "psql -q -d papers -c 'CREATE EXTENSION IF NOT EXISTS vector'"
psql -d papers -At -c "SELECT current_user, extversion FROM pg_extension WHERE extname = 'vector'"

<div style="text-align: justify; max-width: 90ch">

The Python packages, pinned to the versions this notebook was tested with. On your laptop the course installs packages with `uv`;
Kaggle's notebooks install them with `pip`, the installer `uv` replaces.

</div>

In [ ]:
%pip install -q "mlflow==3.17.0" "langchain-core==1.6.3" "langchain-ollama==1.1.0" "langchain-postgres==0.0.17" "langchain-text-splitters==1.1.2" "langgraph==1.2.11" "psycopg[binary]==3.3.5" "pypdf==6.19.0"

In [ ]:
import json
import re
from pathlib import Path
from typing import Literal, TypedDict

import mlflow
import pandas as pd
import psycopg
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_postgres import PGVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from mlflow.entities import SpanType
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import scorer
from pydantic import BaseModel, Field
from pypdf import PdfReader
from sklearn.metrics import confusion_matrix

<div style="text-align: justify; max-width: 90ch">

MLflow writes to a SQLite file instead of a tracking server: Kaggle has no server to keep running, and a file in `/kaggle/working` is saved with the notebook's version,
so anyone can download it (section 9).

</div>

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

<div style="text-align: justify; max-width: 90ch">

## 2. A corpus you may republish

A public notebook shows passages of its papers to anyone, so every paper needs a license that allows republishing. Every ACL Anthology paper since 2016 is CC BY 4.0 ([copyright FAQ](https://aclanthology.org/faq/copyright)).

`papers.csv` records each paper's id, title, authors, year, link and license; the next cell stops the notebook if any license is not on the allowed list.

</div>

In [ ]:
DATA_DIR = next(Path("/kaggle/input").rglob("papers.csv")).parent
papers = pd.read_csv(DATA_DIR / "papers.csv")

not_allowed = papers[~papers["license"].isin(ALLOWED_LICENSES)]
assert not_allowed.empty, f"Licenses that do not allow republishing:\n{not_allowed}"

print(f"{len(papers)} papers; licenses: {papers['license'].value_counts().to_dict()}")
papers[["id", "title", "year"]].head()

<div style="text-align: justify; max-width: 90ch">

A PDF page becomes one LangChain `Document`, with the paper's id, title, link and page number as metadata. Reading stops at the reference list:
titles of other papers answer nothing, and their words would make every paper look relevant to every question.

The chunks are those of `lec_02c`: 800 characters with an overlap of 100.

</div>

In [ ]:
REFERENCES = re.compile(r"^\s*References\s*$", re.MULTILINE)


def read_pages_of(paper):
    """One Document per page of a paper's PDF, up to its reference list."""
    reader = PdfReader(DATA_DIR / f"{paper['id']}.pdf")
    for number, page in enumerate(reader.pages, start=1):
        text = re.sub(
            r"-\n(?=[a-z])", "", page.extract_text() or ""
        )  # rejoin hyphenated words
        references = REFERENCES.search(text)
        if references:
            text = text[: references.start()]
        if text.strip():
            metadata = {
                "source": paper["id"],
                "title": paper["title"],
                "url": paper["url"],
                "page": number,
            }
            yield Document(page_content=text, metadata=metadata)
        if references:
            return


splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100, add_start_index=True
)
chunks = [
    chunk
    for _, paper in papers.iterrows()
    for chunk in splitter.split_documents(list(read_pages_of(paper)))
    if len(chunk.page_content) >= 80  # a lone heading or page number answers nothing
]

print(f"{len(chunks)} chunks from {len(papers)} papers")

<div style="text-align: justify; max-width: 90ch">

## 3. The index in pgvector

The `PGVector` store of `lec_02e`, with one change: the connection goes through PostgreSQL's local socket folder (`host=/var/run/postgresql`) as the role `root`,
so the address holds no password. `create_extension=False`, because `root` may not install extensions; the setup cell already did.

</div>

In [ ]:
SQL_URL = "postgresql://root@/papers?host=/var/run/postgresql"

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
store = PGVector(
    embeddings=embeddings,
    collection_name="papers",
    connection=SQL_URL.replace("postgresql://", "postgresql+psycopg://"),
    use_jsonb=True,
    create_extension=False,
)

COUNT_SQL = """
SELECT count(*) FROM langchain_pg_embedding e
JOIN langchain_pg_collection c ON e.collection_id = c.uuid
WHERE c.name = 'papers'
"""


def count_stored_chunks():
    """Number of chunks in the papers collection."""
    with psycopg.connect(SQL_URL) as connection:
        return connection.execute(COUNT_SQL).fetchone()[0]

<div style="text-align: justify; max-width: 90ch">

Built once per session, and skipped when the count already matches, as in `lec_02e`. On a T4 the embeddings take about a minute in one example run;
on a laptop's CPU the same step takes far longer.

</div>

In [ ]:
if count_stored_chunks() != len(chunks):
    store.delete_collection()  # empty, half-built or out of date: start again
    store.create_collection()
    for start in range(0, len(chunks), 256):
        batch = chunks[start : start + 256]
        ids = [
            f"{c.metadata['source']}#p{c.metadata['page']}#{c.metadata['start_index']}"
            for c in batch
        ]
        store.add_documents(batch, ids=ids)

retriever = store.as_retriever(search_kwargs={"k": TOP_K})
print(f"{count_stored_chunks()} chunks in PostgreSQL")

<div style="text-align: justify; max-width: 90ch">

## 4. Answers that cite their papers

A reader of a literature assistant needs to know which papers an answer comes from. The answer is therefore a pydantic schema (`lec_01f`) with two fields:
the text, and the ids of the papers it used.

`PaperId` lists the 23 ids from `papers.csv` as a `Literal`, so the schema holds an `enum`, which Ollama enforces while it generates (`lec_01c`).
Without it, a small model in testing copied whole excerpt labels, title and page included, instead of the ids.

</div>

In [ ]:
PaperId = Literal[tuple(papers["id"])]  # the only values a citation may take


class CitedAnswer(BaseModel):
    """An answer drawn from paper excerpts, with the papers it used."""

    answer: str = Field(description="the answer, in at most 120 words")
    papers: list[PaperId] = Field(description="ids of the papers the answer uses")


def format_excerpts_from(docs):
    """The retrieved chunks as the text block the prompt receives, each labelled with its paper."""
    return "\n\n".join(
        f"[paper: {d.metadata['source']} | {d.metadata['title']}, page {d.metadata['page']}]\n{d.page_content}"
        for d in docs
    )


ANSWER_PROMPT = ChatPromptTemplate.from_template(
    "You answer questions about research papers on evaluating LLM agents and LLM judges, "
    "using only the excerpts below. If the excerpts do not contain the answer, say that the "
    "papers do not contain it and cite no paper.\n\nExcerpts:\n{context}\n\nQuestion: {question}"
)

<div style="text-align: justify; max-width: 90ch">

One model object per variant, each at temperature 0 with thinking off, as in the course. `num_predict` leaves room for the JSON around a 120-word answer:
an answer cut off in the middle is not valid JSON, and the schema check fails.

</div>

In [ ]:
def make_llm(model):
    """A chat model on the local Ollama server, set up as in the course."""
    return ChatOllama(model=model, reasoning=False, temperature=0, num_predict=600)


answer_chain_for = {
    model: ANSWER_PROMPT
    | make_llm(model).with_structured_output(CitedAnswer, method="json_schema")
    for model in [SMALL_MODEL, LARGE_MODEL]
}

question = "Which two process-level metrics does GEMMAS propose?"
docs = retriever.invoke(question)
cited = answer_chain_for[LARGE_MODEL].invoke(
    {"context": format_excerpts_from(docs), "question": question}
)

print(cited.answer)
print(cited.papers)

<div style="text-align: justify; max-width: 90ch">

Check that the answer names the two metrics and cites the GEMMAS paper; the enum guarantees only that every citation is a real id, not that it is the right one.
The first call of a session also loads the model onto the GPU, so it is the slowest.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. A graph that searches again or refuses

A question in other words than the paper's can miss the right chunks (`lec_03e`). The graph of `lec_03e`, adapted to papers, has five nodes:

- **`rewrite`**: the model writes a search query in the papers' words; after a failed search, a different one
- **`retrieve`**: plain code, the pgvector search
- **`grade`**: the model says whether the excerpts answer the question, fully or in part
- **`answer`**: the cited answer of section 4
- **`refuse`**: plain code, no model; after two failed searches it says that the papers do not contain the answer

</div>

In [ ]:
REWRITE_PROMPT = ChatPromptTemplate.from_template(
    "Rewrite the question below as a short search query for research papers, with the "
    "technical terms a paper would use. Keep every name and number from the question. "
    "Reply with the query only.\n\nQuestion: {question}"
)
RETRY_PROMPT = ChatPromptTemplate.from_template(
    "The search query below did not find the answer to the question. Write another short "
    "search query for the same question, in other words. Reply with the query only."
    "\n\nQuestion: {question}\n\nQuery that failed: {query}"
)
GRADE_PROMPT = ChatPromptTemplate.from_template(
    "Excerpts:\n{context}\n\nQuestion: {question}\n\n"
    "Do the excerpts above contain information that answers the question, fully or in part?"
)


class Grade(BaseModel):
    """Whether retrieved excerpts answer a question."""

    answer_found: Literal["yes", "no"]


chains_for = {
    model: {
        "rewrite": REWRITE_PROMPT | make_llm(model) | StrOutputParser(),
        "retry": RETRY_PROMPT | make_llm(model) | StrOutputParser(),
        "grade": GRADE_PROMPT
        | make_llm(model).with_structured_output(Grade, method="json_schema"),
        "answer": answer_chain_for[model],
    }
    for model in [SMALL_MODEL, LARGE_MODEL]
}

<div style="text-align: justify; max-width: 90ch">

The state carries the model's name, so one graph serves both variants: each node takes its chains from `chains_for[state["model"]]`.

</div>

In [ ]:
MAX_SEARCHES = 2


def rewrite(state):
    """Node: a search query in the papers' words; after a failed search, a different one."""
    chains = chains_for[state["model"]]
    if state.get("searches", 0) == 0:
        query = chains["rewrite"].invoke({"question": state["question"]})
    else:
        query = chains["retry"].invoke(
            {"question": state["question"], "query": state["query"]}
        )
    return {"query": query.strip()}


def retrieve(state):
    """Node: plain code. Search pgvector with the model's query; count the search."""
    return {
        "documents": retriever.invoke(state["query"]),
        "searches": state.get("searches", 0) + 1,
    }


def grade(state):
    """Node: the model says whether the excerpts answer the question."""
    context = format_excerpts_from(state["documents"])
    verdict = chains_for[state["model"]]["grade"].invoke(
        {"context": context, "question": state["question"]}
    )
    return {"grade": verdict.answer_found}


def answer(state):
    """Node: the cited answer, from the excerpts of the latest search."""
    context = format_excerpts_from(state["documents"])
    cited = chains_for[state["model"]]["answer"].invoke(
        {"context": context, "question": state["question"]}
    )
    return {"answer": cited.model_dump()}


def refuse(state):
    """Node: plain code. No search found the answer, so say so and cite nothing."""
    return {
        "answer": {
            "answer": "The papers do not contain the answer to this question.",
            "papers": [],
        }
    }


def route_after_grade(state):
    """Conditional edge: answer, search again, or refuse after the last allowed search."""
    if state["grade"] == "yes":
        return "answer"
    return "refuse" if state["searches"] >= MAX_SEARCHES else "rewrite"

In [ ]:
PaperState = TypedDict(
    "PaperState",
    {
        "model": str,  # which variant answers
        "question": str,  # the user's question, never changed
        "query": str,  # the search query the model wrote
        "documents": list,  # the chunks of the latest search
        "searches": int,  # how many searches so far
        "grade": str,  # "yes" or "no"
        "answer": dict,  # a CitedAnswer as a dict
    },
)

builder = StateGraph(PaperState)
for name, node in [
    ("rewrite", rewrite),
    ("retrieve", retrieve),
    ("grade", grade),
    ("answer", answer),
    ("refuse", refuse),
]:
    builder.add_node(name, node)
builder.add_edge(START, "rewrite")
builder.add_edge("rewrite", "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges(
    "grade", route_after_grade, ["answer", "rewrite", "refuse"]
)
builder.add_edge("answer", END)
builder.add_edge("refuse", END)
graph = builder.compile()

pd.DataFrame(
    [
        {"from": e.source, "to": e.target, "conditional": e.conditional}
        for e in graph.get_graph().edges
    ]
)

<div style="text-align: justify; max-width: 90ch">

One question the papers cannot answer, step by step: each row is what one node changed.

</div>

In [ ]:
steps = []
for update in graph.stream(
    {
        "model": LARGE_MODEL,
        "question": "How much electricity does it take to train a large language model?",
    },
    stream_mode="updates",
):
    for node, changes in update.items():
        papers_found = sorted(
            {d.metadata["source"] for d in changes.get("documents", [])}
        )
        steps.append(
            {
                "node": node,
                "query": changes.get("query", ""),
                "papers found": papers_found or "",
                "grade": changes.get("grade", ""),
                "answer": changes.get("answer", {}).get("answer", ""),
            }
        )

pd.DataFrame(steps)

<div style="text-align: justify; max-width: 90ch">

Read the `grade` column. Two `no` grades lead to `refuse`, which answers without a model call. If a grade said `yes`, the model judged some excerpt relevant;
then read the answer: does it admit that the papers do not say, or does it invent a number?

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Two variants on the same questions

`eval_set.jsonl` holds 18 questions written by hand from the papers: 16 with the paper that answers them and the facts a correct answer must state, and 2 that no paper answers.
The two variants are the two answer models; the rest of the graph is the same.

The judge reads the whole `expectations` of a record. With the paper's id in it, the judge in testing marked right answers `no` for not naming the id.
So `expectations` holds only the facts, and the scorers look up the expected paper by question.

</div>

In [ ]:
eval_set = pd.read_json(DATA_DIR / "eval_set.jsonl", lines=True)
records = [
    {
        "inputs": {"question": row.question},
        "expectations": {"expected_facts": row.expected_facts},
    }
    for row in eval_set.itertuples()
]
PAPER_OF = dict(zip(eval_set["question"], eval_set["expected_paper"]))

print(
    f"{len(records)} questions, {eval_set['expected_paper'].isna().sum()} without an answer in the papers"
)

<div style="text-align: justify; max-width: 90ch">

Four deterministic scorers, read from the answer or from the trace:

- **`retrieval_hit`**: did any search find a chunk of the expected paper?
- **`cites_expected_paper`**: does the answer cite it?
- **`refused`**: did the answer cite no paper at all?
- **`searches`**: how many searches the graph ran; more searches mean more time

The first two abstain (`None`) on the questions no paper answers.

</div>

In [ ]:
def found_papers_in(trace):
    """Ids of the papers in every RETRIEVER span of the trace."""
    spans = trace.search_spans(span_type=SpanType.RETRIEVER)
    return {doc["metadata"]["source"] for span in spans for doc in (span.outputs or [])}


@scorer(aggregations=["mean"])
def retrieval_hit(inputs, trace):
    expected = PAPER_OF[inputs["question"]]
    return None if pd.isna(expected) else expected in found_papers_in(trace)


@scorer(aggregations=["mean"])
def cites_expected_paper(inputs, outputs):
    expected = PAPER_OF[inputs["question"]]
    return None if pd.isna(expected) else expected in outputs["papers"]


@scorer(aggregations=["mean"])
def refused(outputs):
    return not outputs["papers"]


@scorer(aggregations=["mean"])
def searches(trace):
    return len(trace.search_spans(span_type=SpanType.RETRIEVER))

<div style="text-align: justify; max-width: 90ch">

The judge of `lec_03b`, with its rubric, on the larger `qwen3:14b`: a judge should not grade its own answers, and section 7 checks whether this one is good enough.

</div>

In [ ]:
FACTS_RUBRIC = (
    "Question: {{ inputs }}\n\n"
    "Answer: {{ outputs }}\n\n"
    "Expected facts: {{ expectations }}\n\n"
    "Reply yes if the answer states every expected fact, in any wording, "
    "and contradicts none of them. "
    "Reply no if any expected fact is missing or the answer states its opposite."
)

states_facts = make_judge(
    name="states_facts",
    instructions=FACTS_RUBRIC,
    model=f"ollama:/{JUDGE_MODEL}",
    feedback_value_type=Literal["yes", "no"],
    inference_params={"temperature": 0, "reasoning_effort": "none"},
    base_url="http://localhost:11434/v1/chat/completions",
)
states_facts.aggregations = ["mean"]

SCORERS = [retrieval_hit, cites_expected_paper, refused, searches, states_facts]

<div style="text-align: justify; max-width: 90ch">

One MLflow run per variant, on the same records and scorers. Each run stores its metrics, and each answer its trace, in `mlflow.db`.

</div>

In [ ]:
def make_predict(model):
    """The predict_fn of one variant: the graph with this model, returning the cited answer."""

    def predict(question):
        return graph.invoke({"model": model, "question": question})["answer"]

    return predict


results_by_variant = {}
for model in [SMALL_MODEL, LARGE_MODEL]:
    with mlflow.start_run(run_name=model):
        results_by_variant[model] = mlflow.genai.evaluate(
            data=records, predict_fn=make_predict(model), scorers=SCORERS
        )

In [ ]:
comparison = {}
for model, results in results_by_variant.items():
    column = dict(results.metrics)
    column["seconds_per_answer/mean"] = (
        results.result_df["execution_duration"].mean() / 1000
    )  # ms to s
    comparison[model] = column

pd.DataFrame(comparison).sort_index().round(2)

<div style="text-align: justify; max-width: 90ch">

Read the table as a trade-off, as in `lec_03b`:

- **`retrieval_hit` and `cites_expected_paper`**: a hit that is not cited means the right chunk reached the model, which then cited another paper or none.
- **`states_facts`**: the share of `yes` verdicts; worth comparing only if section 7 finds the judge good enough.
- **`seconds_per_answer` and `searches`**: what the better answers cost in time.

With 18 questions, one question is about 6 points: a small difference may be noise.

</div>

In [ ]:
def build_question_table_for(results):
    """One row per question id: the verdict, the citation check and the searches."""
    frame = results.result_df
    # "request" is the inputs dict, or the plain question, depending on the trace
    questions = [r["question"] if isinstance(r, dict) else r for r in frame["request"]]
    ids = eval_set.set_index("question").loc[questions, "id"].to_numpy()
    columns = ["states_facts", "cites_expected_paper", "refused", "searches"]
    return pd.DataFrame(
        {c: frame[f"{c}/value"].to_numpy() for c in columns}, index=ids
    ).sort_index()


pd.concat(
    {model: build_question_table_for(r) for model, r in results_by_variant.items()},
    axis=1,
)

<div style="text-align: justify; max-width: 90ch">

Read q17 and q18 first, the questions no paper answers: `refused` should be `True` for both variants. Then look for `refused` on q01 to q16:
a refusal there means two grades said `no` to a question the papers answer.

Finally, find the rows where the variants disagree on `states_facts` and read those answers in the MLflow UI (section 9):
which variant missed a fact, and was the fact in the excerpts?

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. Is the judge good enough?

`judge_labels.jsonl` holds 14 fixed answers, each labelled `yes` or `no` by hand against the rubric: real answers from earlier runs and answers written with a known flaw.
Half are labelled `no` on purpose: a set of right answers only would never catch a judge that says `yes` to everything (`lec_03c`).

</div>

<div style="text-align: justify; max-width: 90ch">

`lec_03c` called the judge in a loop and kept the verdicts in a table. Here the check is an MLflow evaluation of its own,
so every verdict is a trace in `mlflow.db`, with the judge's rationale, next to the runs of the two variants:

- **Fixed answers**: each record already holds its `outputs`, so `evaluate` runs without a `predict_fn`.
- **Labels kept out**: the judge reads the whole `expectations`, so the hand label stays in the table, not in the record.

</div>

In [ ]:
labelled = pd.read_json(DATA_DIR / "judge_labels.jsonl", lines=True)
records_by_id = dict(zip(eval_set["id"], records))
check_records = [
    {
        **records_by_id[row.question_id],
        "outputs": {"answer": row.answer, "papers": row.papers},
    }
    for row in labelled.itertuples()
]


def answer_text_of(response):
    """The answer text of a result row's response: a dict, or the dict as JSON text."""
    if isinstance(response, str):
        response = json.loads(response)
    return response["answer"]

<div style="text-align: justify; max-width: 90ch">

One run, `judge-check`. The verdicts come back in `result_df`, matched to the labels by answer text; the agreement and the two error rates are logged to the same run as metrics.

</div>

In [ ]:
with mlflow.start_run(run_name="judge-check"):
    check = mlflow.genai.evaluate(data=check_records, scorers=[states_facts])
    frame = check.result_df
    verdict_of = dict(
        zip(
            frame["response"].map(answer_text_of),
            frame["states_facts/value"].astype(str),
        )
    )
    labelled = labelled.assign(verdict=labelled["answer"].map(verdict_of))

    labels, judged = labelled["label"], labelled["verdict"]
    rates = {
        "agreement": (labels == judged).mean(),
        "false_yes_rate": (
            judged[labels == "no"] == "yes"
        ).mean(),  # passed answers labelled no
        "false_no_rate": (
            judged[labels == "yes"] != "yes"
        ).mean(),  # failed answers labelled yes
    }
    mlflow.log_metrics(rates)

labelled[["id", "kind", "label", "verdict", "why"]]

In [ ]:
print({name: round(float(value), 2) for name, value in rates.items()})

pd.DataFrame(
    confusion_matrix(labels, judged, labels=["yes", "no"]),
    index=["label yes", "label no"],
    columns=["judge yes", "judge no"],
)

<div style="text-align: justify; max-width: 90ch">

The lower-left cell counts the dangerous errors: answers labelled `no` that the judge passed. A lenient judge makes every variant look good, so its `states_facts` cannot separate them.

For every disagreement, read the answer and the `why`, and the judge's rationale in the `judge-check` run (section 9): is the judge wrong, or is the label?
Writing the labels is also how you find questions whose facts are ambiguous.

</div>

<div style="text-align: justify; max-width: 90ch">

## 8. Decision record

The model choice, requirements first, as in `lec_01b`:

| Requirement | This assistant |
| --- | --- |
| Task | Answer questions about 23 papers, citing them |
| Languages | English questions, English papers |
| Context | 4 chunks of 800 characters, about 1,000 tokens per prompt |
| Latency | Seconds per answer are fine: one reader, not a crowd |
| Hardware | One Kaggle T4, 15 GB of GPU memory, free |
| License | Qwen3 models are Apache 2.0 |
| Privacy | The papers are public, so running on Kaggle's servers is acceptable |

**Candidates**: `qwen3:1.7b` and `qwen3:8b`, the course's two tiers; both fit one T4.

</div>

<div style="text-align: justify; max-width: 90ch">

**Decision**: choose by the evaluation in this run, in this order:

- **The judge first**: if its false-yes rate in section 7 is high, decide on `cites_expected_paper` and `refused` alone.
- **Quality next**: choose the variant with the higher `states_facts` and `cites_expected_paper` when the gap is larger than one or two questions.
- **Cost last**: with no clear gap, choose the faster variant.

In your own notebook, write the decision with the numbers of your saved version.

</div>

<div style="text-align: justify; max-width: 90ch">

## 9. Share the results

The saved version of this notebook is the public record: every table above, as it came out of this run. The traces, with every prompt, chunk and answer, are in `mlflow.db`, which Kaggle keeps as an output of the version.

To explore them, download `mlflow.db` from the version's *Output* tab and open it from the course folder:

```bash
uv run mlflow server --backend-store-uri sqlite:///mlflow.db --port 5011
```

</div>

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
size_mb = Path(MLFLOW_DB).stat().st_size / 1e6

print(f"mlflow.db: {size_mb:.1f} MB, {len(runs)} runs")
runs.filter(regex=r"runName$|/mean$|agreement$|_rate$")

<div style="text-align: justify; max-width: 90ch">

## Recap

- **Kaggle** gives a free GPU and a fresh machine per session: the notebook installs Ollama, PostgreSQL with pgvector and its packages at the top.
- **Peer authentication** connects to PostgreSQL with no password at all.
- A **pydantic** schema makes every answer cite its papers, which a scorer can check.
- The **graph** searches again in other words, and refuses when no search finds the answer.
- **MLflow** compares two variants on the same questions, and the judge is trusted only after it agrees with hand labels.

**Next**: the assignment's README lists what your own version must contain.

</div>